# পাঠ 10 - উৎপাদনে AI এজেন্ট

এই পাঠে আপনি Microsoft Agent Framework (Python) ব্যবহার করে AI এজেন্টদের জন্য **উৎপাদন প্যাটার্ন** শিখবেন। আমরা আলোচনা করব:

- **Observability** — এজেন্ট ইন্টারঅ্যাকশনে টাইমিং এবং লগিং যোগ করা
- **Evaluation** — একটি মূল্যায়নকারী এজেন্ট ব্যবহার করে প্রতিক্রিয়ার গুণমান স্কোর করা
- **Cost management** — টোকেন অপ্টিমাইজেশন এবং মডেল নির্বাচন সংক্রান্ত কৌশল

সিনারিওটি একটি **ভ্রমণ এজেন্ট** যা ব্যবহারকারীদের ভ্রমণের পরিকল্পনা করতে সাহায্য করে, এবং যাতে পর্যবেক্ষণ ও মূল্যায়ন স্তরযুক্ত।


## সেটআপ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## উৎপাদনে বিবেচ্য বিষয়

প্রোটোটাইপ থেকে উৎপাদনে AI এজেন্টগুলো নিয়ে যাওয়ার জন্য তিনটি স্তম্ভে সতর্ক মনোযোগ প্রয়োজন:

1. **পর্যবেক্ষণযোগ্যতা** — আপনাকে এজেন্ট কী করছে, এটি কতক্ষণ নিচ্ছে, এবং এটি কোন টুলগুলো কল করছে—এসব বিষয়ে দৃশ্যমানতা থাকতে হবে। ট্রেসিং এবং লগিং ছাড়া উৎপাদনের সমস্যাগুলো ডিবাগ করা প্রায় অসম্ভব।

2. **মূল্যায়ন** — স্বয়ংক্রিয় মানের পরীক্ষা নিশ্চিত করে যে এজেন্টের প্রতিক্রিয়াগুলি সময়ের সাথে সঠিক, সম্পূর্ণ, এবং সহায়ক থাকে। একটি মূল্যায়নকারী এজেন্ট নির্ধারিত মানদণ্ড অনুযায়ী প্রতিক্রিয়াগুলিকে স্কোর দিতে পারে।

3. **খরচ ব্যবস্থাপনা** — টোকেন ব্যবহারে সরাসরি খরচ প্রভাবিত হয়। প্রম্পট অপ্টিমাইজেশন, মডেল নির্বাচন, এবং ক্যাশিংয়ের মতো কৌশলগুলো মান নষ্ট না করে খরচ নিয়ন্ত্রণে রাখতে সাহায্য করে।


## একটি পর্যবেক্ষণযোগ্য এজেন্ট তৈরি করা

আমরা ভ্রমণ সরঞ্জাম সংজ্ঞায়িত করি এবং এজেন্ট কলকে টাইমিং দিয়ে আবৃত করি যাতে আমরা বিলম্ব নিরীক্ষণ করতে পারি। প্রোডাকশনে আপনি OpenTelemetry বা অনুরূপ ট্রেসিং ব্যাকএন্ডের সাথে ইন্টিগ্রেট করবেন।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## মূল্যায়ন প্যাটার্নসমূহ

একটি সাধারণ প্রোডাকশন প্যাটার্ন হলো দ্বিতীয় একটি এজেন্টকে **মূল্যায়ক** হিসেবে ব্যবহার করা। মূল্যায়ক প্রাথমিক এজেন্টের উত্তরকে পূর্বনির্ধারিত মানদণ্ড যেমন সম্পূর্ণতা, সঠিকতা, এবং সহায়কতার বিরুদ্ধে স্কোর করে।

এটি সক্ষম করে:
- ব্যবহারকারীদের কাছে উত্তর পৌঁছানোর আগে স্বয়ংক্রিয় মানগত চেক
- প্রম্পট বা মডেল পরিবর্তিত হলে রিগ্রেশন সনাক্তকরণ
- সময়ের সাথে এজেন্টের কর্মদক্ষতার ধারাবাহিক পর্যবেক্ষণ


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## খরচ ব্যবস্থাপনা কৌশল

উৎপাদন-পর্যায়ের AI এজেন্টদের জন্য খরচ নিয়ন্ত্রণ অত্যন্ত গুরুত্বপূর্ণ। এখানে মূল কৌশলসমূহ:

| কৌশল | বর্ণনা |
|---|---|
| **প্রম্পট অপ্টিমাইজেশন** | সিস্টেম নির্দেশনাগুলো সংক্ষিপ্ত রাখুন। ইনপুট টোকেন কমানোর জন্য অনাবশ্যক প্রসঙ্গগুলো সরান। |
| **মডেল নির্বাচন** | অত্যন্ত সহজ কাজগুলোর জন্য ছোট, সস্তা মডেল (উদাহরণস্বরূপ GPT-4o-mini) ব্যবহার করুন যেমন শ্রেণীকরণ বা তথ্য আহরণ, এবং জটিল যুক্তির জন্য বড় মডেল সংরক্ষণ করুন। |
| **ক্যাশিং** | পুনরাবৃত্তি API কল এড়াতে টুলের ফলাফল এবং ঘন ঘন অনুরোধগুলো ক্যাশ করুন। |
| **টোকেন বাজেট** | অপ্রত্যাশিতভাবে দীর্ঘ উত্তর প্রতিরোধ করার জন্য `max_tokens` সীমা নির্ধারণ করুন। |
| **ব্যাচিং** | সম্ভব হলে একাধিক ব্যবহারকারীর প্রশ্নকে একটি API কল-এ গ্রুপ করুন। |

প্রায়োগিকভাবে, স্তরভিত্তিক পদ্ধতি ভাল কাজ করে: সরল অনুরোধগুলো দ্রুত এবং সস্তা মডেলে রুট করুন এবং শুধুমাত্র জটিল প্রশ্নগুলোকে আরও ক্ষমতাবান মডেলে উন্নীত করুন।


## সারাংশ

এই পাঠে আপনি কীভাবে নিম্নলিখিত কাজগুলো করতে হয় তা শিখেছেন:

1. **এজেন্ট ইন্টারঅ্যাকশনগুলিতে পর্যবেক্ষণ যোগ করা** সময় নিরীক্ষণ এবং লগিং-এর মাধ্যমে, ট্রেসিং এবং মনিটরিংয়ের জন্য ভিত্তি তৈরি করা।
2. **এজেন্টের প্রতিক্রিয়াগুলো মূল্যায়ন করা** স্বয়ংক্রিয়ভাবে একটি মূল্যায়নকারী এজেন্ট ব্যবহার করে যা সম্পূর্ণতা, নির্ভুলতা, এবং সহায়কতার স্কোর নির্ধারণ করে।
3. **খরচ নিয়ন্ত্রণ করা** প্রম্পট অপ্টিমাইজেশন, মডেল নির্বাচন, ক্যাশিং, এবং টোকেন বাজেটের মাধ্যমে।

এই প্রোডাকশন প্যাটার্নগুলো নিশ্চিত করতে সাহায্য করে যে আপনার AI এজেন্টগুলো বৃহৎ পরিসরে নির্ভরযোগ্য, পরিমাপযোগ্য, এবং খরচ-সাশ্রয়ী।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
অস্বীকারোক্তি:
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনুবাদ করা হয়েছে। আমরা যথাযথতা নিশ্চিত করতে চেষ্টা করি, তবু স্বয়ংক্রিয় অনুবাদে ত্রুটি বা ভুল থাকার সম্ভাবনা থাকতে পারে। মূল নথিটি তার নিজ ভাষায়ই কর্তৃপক্ষমূলক উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদের পরামর্শ দেয়া হয়। এই অনুবাদের ব্যবহারের ফলে যে কোনো ভুল বোঝাবুঝি বা মিসইন্টারপ্রিটেশনের জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
